# 08-phase9-group-graph-foundations

**neuron Phase 9** — Hidden layer 자체를 graph 로 (group-as-node, 사용자 vision 의 architecture
구성 계획 B 진입).

핵심 가설:
1. **function preservation** — GroupGraphLinear(adj=full) 가 standard Linear 와 forward 동치?
2. **adjacency 학습** — adj 파라미터에 gradient 가 흘러 routing 이 학습됨?
3. **identity init (block-diagonal)** 의 inductive bias — 시작부터 group-wise 독립 학습 vs full 보다 빠르거나 느림?
4. **adjacency 시각화** — 학습 후 어떤 group 들이 서로 강하게 연결되는가?

설계: 3-way sweep × 2 seed = 6 run, max_steps=1500.
- arch ∈ {plain, group_full, group_identity}
- seed ∈ {42, 123}

데이터: TinyShakespeare (char-LM)
시드: [42, 123]
작성일: 2026-05-26
연관: Issue [#59](https://github.com/EinSofINTEREST/GraphLM/issues/59) / Phase 8 baseline PR [#56](https://github.com/EinSofINTEREST/GraphLM/pull/56) / [아키텍처 구성 계획 (Notion)](https://www.notion.so/36ce8b70b7aa818cbf1fe71687b449b8)

## 1. 환경

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.graph_group_demo import train_group_graph_mlp
from graphlm.utils import repo_root

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)
print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)

## 2. 실험 설정

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase9"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123]
ARCHS = ["plain", "group_full", "group_identity"]
EMB_DIM = 64
HIDDEN_DIM = 256  # group_size 16 의 배수
GROUP_SIZE = 16  # → n_groups_in/out = 16 (256/16) — 시각화 적합
N_GRAM = 4
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500

# vocab*N_GRAM*EMB_DIM = N_GRAM * EMB_DIM = 256 = HIDDEN_DIM = vocab_logits 입력
# vocab_size 는 65 라 GROUP_SIZE 의 배수가 아님 — group_*는 vocab 도 padding 또는 unequal split
# 단순화: vocab_size 도 GROUP_SIZE 배수가 되도록 pad — 노트북에서 wrapper 처리

print(f"SEEDS      = {SEEDS}")
print(f"ARCHS      = {ARCHS}")
print(
    f"HIDDEN_DIM = {HIDDEN_DIM}, GROUP_SIZE = {GROUP_SIZE} → n_groups = {HIDDEN_DIM // GROUP_SIZE}"
)
print(f"MAX_STEPS  = {MAX_STEPS}")
print(f"전체 run   = {len(SEEDS) * len(ARCHS)}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
V = tokenizer.vocab_size

# vocab_size 가 GROUP_SIZE 의 배수가 아니면 group_*는 fc2 의 out_features 가 안 맞음.
# 노트북 단순화: vocab_size 를 GROUP_SIZE 배수로 padding (예: 65 → 80) 한 가상 vocab 사용
import math

V_PADDED = math.ceil(V / GROUP_SIZE) * GROUP_SIZE
print(f"vocab_size : {V}, padded to {V_PADDED} (GROUP_SIZE {GROUP_SIZE} 배수)")

## 4. sweep 학습

각 (seed, arch) 에 대해 1 run. plain 은 baseline, group_full 은 function-preserving 시작,
group_identity 는 block-diagonal sparse 시작.

In [ ]:
runs = {}
for seed in SEEDS:
    for arch in ARCHS:
        key = (seed, arch)
        print(f"--- seed={seed}, arch={arch} ---")
        runs[key] = train_group_graph_mlp(
            dataset=dataset,
            vocab_size=V_PADDED,
            seed=seed,
            arch=arch,
            emb_dim=EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            group_size=GROUP_SIZE,
            n_gram=N_GRAM,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            device=DEVICE,
        )
        print(f"  done: final_loss={runs[key]['final_loss']:.4f}")
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 — arch × seed

In [ ]:
import statistics

print(f"{'arch':>16}  {'seed':>5}  {'final_loss':>11}")
print("-" * 40)
for arch in ARCHS:
    for seed in SEEDS:
        r = runs[(seed, arch)]
        print(f"{arch:>16}  {seed:>5}  {r['final_loss']:>11.4f}")

print()
print("=== arch 별 mean ===")
for arch in ARCHS:
    fls = [runs[(s, arch)]["final_loss"] for s in SEEDS]
    print(f"  {arch:>16}: mean={statistics.mean(fls):.4f}, range={max(fls) - min(fls):.4f}")

## 6. adjacency 학습 진화 — fc1 / fc2 의 학습된 adj heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1])

# row 0: group_full 의 fc1, fc2 (seed 42)
# row 1: group_identity 의 fc1, fc2 (seed 42)
for row_i, arch in enumerate(["group_full", "group_identity"]):
    r = runs[(SEEDS[0], arch)]
    if r["final_adj"] is None:
        continue
    for col_i, layer_name in enumerate(["fc1", "fc2"]):
        ax = fig.add_subplot(gs[row_i, col_i * 2 : col_i * 2 + 2])
        adj = r["final_adj"][layer_name].numpy()
        vmax = max(abs(adj).max(), 1e-6)
        im = ax.imshow(adj, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(f"{arch} — {layer_name} adj (seed={SEEDS[0]})  shape={adj.shape}")
        ax.set_xlabel("group_in")
        ax.set_ylabel("group_out")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(OUT_DIR / "adjacency_heatmaps.png", dpi=120)
plt.show()

## 7. loss curve 비교 (arch 별 mean ± σ across 2 seeds)

In [ ]:
window = 30
colors = {"plain": "#1f77b4", "group_full": "#2ca02c", "group_identity": "#ff7f0e"}

fig, ax = plt.subplots(figsize=(13, 5))
for arch in ARCHS:
    seed_curves = []
    for seed in SEEDS:
        losses = runs[(seed, arch)]["losses"]
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        seed_curves.append(smoothed)
    arr = np.array(seed_curves)
    steps = np.arange(window - 1, window - 1 + arr.shape[1])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0, ddof=1)
    ax.plot(steps, mean, color=colors[arch], lw=1.5, label=arch)
    ax.fill_between(steps, mean - std, mean + std, color=colors[arch], alpha=0.15)
ax.set_xlabel("step")
ax.set_ylabel(f"loss (smoothed window={window})")
ax.set_title(f"Phase 9 — plain Linear vs GroupGraphLinear (mean ± σ over {len(SEEDS)} seeds)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curves.png", dpi=120)
plt.show()

## 결과 요약 / Phase 10 권장 방향

확인 포인트:
- §5 final_loss — group_full 이 plain 과 비슷? (function preservation 가설 입증)
- §5 group_identity vs plain — block-diagonal sparse init 의 학습 성능 (inductive bias 효과)
- §6 adjacency heatmap — group_full 학습 후 어떤 group 간 연결이 강해졌나? group_identity 의 off-diagonal 학습 활성도?
- §7 loss curve — 세 arch 의 수렴 속도 비교

**판정 시나리오**:
- **A. group_full ≈ plain** ⭐ — function preservation 입증. group routing 학습 가능성 확인
- **B. group_full < plain** — adjacency 학습이 plain Linear 보다 능력 강화 (drop-in 대체 후보)
- **C. group_identity ≈ group_full** — block-diagonal sparse init 도 충분 (Phase 10 의 sparsification 권장)
- **D. group_identity 명확 열위** — sparse 시작이 학습 능력 제한 — adjacency growth 메커니즘 필요

**참고**:
- 아키텍처 구성 계획 (Notion): https://www.notion.so/36ce8b70b7aa818cbf1fe71687b449b8
- ML 용어집: https://www.notion.so/36ce8b70b7aa812298bbe1388e61b753